# Brave Search API — value comparison demo

A walkthrough comparing three ways to answer questions about recent events:

| Pipeline | Approach |
| --- | --- |
| **baseline** | LLM only, no retrieval |
| **diy_rag** | Manual RAG — search → fetch → extract → chunk → embed → index → retrieve → LLM |
| **brave-search-api** | Brave **LLM Context** endpoint → LLM with citations |

It weighs two questions side by side:

1. **Quality:** Does grounding (`brave-search-api`) reduce hallucinations on recent-events questions, where the ungrounded `baseline` has nothing reliable to draw on?
2. **Infrastructure:** How much code, latency, and dependency overhead does building retrieval by hand (`diy_rag`) add over Brave's single LLM Context call?

LangFuse traces every call, scores every output, and renders the three pipelines side by side at the end.

## 1. Setup

The notebook talks to a local, self-hosted LangFuse deployment via Docker. On first boot LangFuse seeds a project and a fixed key pair, so the `LANGFUSE` values are defaulted for you in the next cell.

You only need to add the following environment variables:

* `BRAVE_API_KEY`
* `ANTHROPIC_API_KEY`

In [ ]:
%env BRAVE_API_KEY=     
%env ANTHROPIC_API_KEY=
%env LANGFUSE_HOST=http://localhost:3005
%env LANGFUSE_PUBLIC_KEY=pk-lf-local-brave-demo
%env LANGFUSE_SECRET_KEY=sk-lf-local-brave-demo

In [2]:
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

from langfuse import get_client

langfuse = get_client()
print(f"✓ LangFuse client ready  (host: {os.environ.get('LANGFUSE_HOST', 'http://localhost:3005')})")

✓ LangFuse client ready  (host: http://localhost:3005)


## 2. The three pipelines

Each pipeline lives in its own file under `pipelines/` behind the same `question → {answer, sources}` contract, so they're interchangeable. The gap in implementation size between them is itself part of the comparison.

In [3]:
# Three pipelines, same interface: takes a question string, returns
# {"answer": str, "sources": list[dict], ...}
from pipelines import (
    baseline_pipeline,
    brave_search_api_pipeline,
    diy_rag,  # for pre-warming the embedder later
    diy_rag_pipeline,
)

print("✓ Pipelines imported")

✓ Pipelines imported


In [4]:
# Line counts (the value prop in a directory listing)
import pathlib

for name in ("baseline.py", "diy_rag.py", "brave_search_api.py"):
    lines = len(pathlib.Path("pipelines", name).read_text().splitlines())
    print(f"  {name:<20} {lines:>4} lines")

  baseline.py            20 lines
  diy_rag.py            294 lines
  brave_search_api.py    86 lines


## 3. One question, three pipelines, in parallel

Before the full evaluation, each pipeline answers the same question — run concurrently, so the answers and latencies line up for a direct read.

The local embedder is pre-warmed *outside* the timed run, so `diy_rag`'s reported latency reflects steady-state performance rather than the one-time model load.

In [5]:
DEMO_QUESTION = (
    "What was the Federal Reserve's most recent interest rate decision in 2026, "
    "and what reasoning did Powell give?"
)
print(f"Question: {DEMO_QUESTION}")

Question: What was the Federal Reserve's most recent interest rate decision in 2026, and what reasoning did Powell give?


In [6]:
# Pre-warm — this is setup, not the timed run. First call downloads ~80MB,
# subsequent calls are cached. Print "done" when complete.
print("Pre-warming local embedder...", end=" ", flush=True)
t0 = time.perf_counter()
diy_rag._get_embedder()
print(f"done in {time.perf_counter() - t0:.1f}s")

Pre-warming local embedder... 

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

done in 2.4s


In [7]:
# Run all three pipelines IN PARALLEL on the same question.
# Wall-clock time = the slowest pipeline (DIY), not the sum.


def run_one(name, fn, question):
    start = time.perf_counter()
    try:
        result = fn(question)
        result["_ok"] = True
    except Exception as e:
        result = {"answer": f"ERROR: {e}", "sources": [], "_ok": False}
    result["_latency_s"] = time.perf_counter() - start
    result["_name"] = name
    return result


pipelines_list = [
    ("baseline", baseline_pipeline),
    ("diy_rag", diy_rag_pipeline),
    ("brave-search-api", brave_search_api_pipeline),
]

wall_start = time.perf_counter()
results = {}
with ThreadPoolExecutor(max_workers=3) as pool:
    futures = {pool.submit(run_one, n, fn, DEMO_QUESTION): n for n, fn in pipelines_list}
    for fut in as_completed(futures):
        r = fut.result()
        results[r["_name"]] = r
        print(f"  ✓ {r['_name']:<16} returned at +{r['_latency_s']:.1f}s")

print(f"\nWall-clock (parallel): {time.perf_counter() - wall_start:.1f}s")

  ✓ baseline         returned at +4.9s
  ✓ diy_rag          returned at +8.1s
  ✓ brave-search-api returned at +8.3s

Wall-clock (parallel): 8.4s


In [8]:
# Display each answer
for name in ("baseline", "diy_rag", "brave-search-api"):
    r = results[name]
    print("=" * 70)
    print(f"{name.upper()}  ({r['_latency_s']:.1f}s)")
    print("=" * 70)
    print(r["answer"])
    if r.get("sources"):
        print(f"\n  Sources cited: {len(r['sources'])}")
    print()

BASELINE  (4.9s)
I don't have reliable information about Federal Reserve decisions in 2026 or any specific statements Chair Powell may have made about them. I can't confirm dates, rate levels, or reasoning without risking inaccuracy.

For the most recent FOMC decision and Powell's accompanying remarks, I'd recommend checking:

- **federalreserve.gov** — official FOMC statements, press conference transcripts, and the Summary of Economic Projections
- Major financial news outlets (Reuters, Bloomberg, WSJ, FT) for analysis and context

If you can share what you've already seen reported, I'm happy to help you interpret it.

DIY_RAG  (8.1s)
At its most recent meeting on March 18, 2026, the Federal Open Market Committee held interest rates unchanged at a target range of 3.50% to 3.75% [2]. This followed three rate cuts totaling a full percentage point in late 2024 [1]. Powell's stated reasoning was that the Fed was in "no hurry" to ease further, citing fresh uncertainty from tariffs, immigra

What to look for above:

- **baseline** — hedges or asserts unverifiable specifics, with no sources to back them.
- **diy_rag** — grounded, with inline `[1]`/`[2]` citations, but slower and exposed to chunking and extraction edge cases.
- **brave-search-api** — equally grounded and cited, with specific facts, on far less machinery.

The next section quantifies these differences across a small dataset with LangFuse scoring.

## 4. Push the evaluation dataset to LangFuse

Five finance questions across distinct categories — central bank, earnings, macro data, corporate, and a deliberately unanswerable "refusal" case. Small enough to run live, varied enough to be representative.

In [9]:
DATASET_NAME = "brave-finance-eval"

QUESTIONS = [
    {
        "question": "What was the Federal Reserve's most recent interest rate decision in 2026, and what reasoning did Powell give?",
        "category": "central_bank",
    },
    {
        "question": "What were Nvidia's most recent quarterly earnings results — revenue, EPS, and guidance?",
        "category": "earnings",
    },
    {
        "question": "What was the most recent US CPI inflation reading released in 2026?",
        "category": "macro_data",
    },
    {
        "question": "What major tech-sector M&A deals were announced in early 2026?",
        "category": "corporate",
    },
    {
        "question": "What will the Federal Reserve decide at its next meeting after May 2026?",
        "category": "refusal_test",
    },
]

try:
    langfuse.create_dataset(
        name=DATASET_NAME,
        description="Finance Q&A on recent events. baseline vs diy_rag vs brave-search-api.",
    )
    print(f"Created dataset '{DATASET_NAME}'.")
except Exception as e:
    print(f"Dataset already exists (or: {e}). Continuing.")

for q in QUESTIONS:
    langfuse.create_dataset_item(
        dataset_name=DATASET_NAME,
        id=q[
            "category"
        ],  # stable id -> idempotent: re-running keeps the dataset at 5, no duplicates
        input={"question": q["question"]},
        metadata={"category": q["category"]},
    )

langfuse.flush()
print(f"Uploaded {len(QUESTIONS)} items.")

Created dataset 'brave-finance-eval'.
Uploaded 5 items.


## 5. Evaluation policy — what is measured

Every dataset item gets three scores:

- **`citation_rate`** (0–1) — LLM-as-judge: the fraction of factual claims carrying an inline citation.
- **`factuality`** (1–5) — LLM-as-judge: how substantive the answer is and how well its claims are grounded in the cited sources.
- **`latency_ms`** — measured during execution.

The judges and task wrappers live in `evaluation.py` as library code, keeping this notebook focused on the workflow. The citation-rate judge prompt is shown below for reference.

In [10]:
from evaluation import (
    EVALUATORS,  # the three evaluators packaged up
    baseline_task,  # task wrappers (handle latency tracking)
    brave_search_api_task,
    diy_task,
)

print(f"✓ Loaded {len(EVALUATORS)} evaluators: " + ", ".join(e.__name__ for e in EVALUATORS))

✓ Loaded 3 evaluators: citation_rate_evaluator, factuality_evaluator, latency_evaluator


In [11]:
# Show what the citation judge actually does. This is the policy
# that determines our headline numbers — worth showing the audience.
import inspect

from evaluation import JUDGE_CITATION

print("Citation rate judge prompt:")
print("-" * 70)
print(JUDGE_CITATION)

Citation rate judge prompt:
----------------------------------------------------------------------
You are evaluating an AI assistant's answer to a finance question.

QUESTION: {question}

ANSWER: {answer}

Step 1: Count distinct factual claims in the answer. A "factual claim" is a specific number, date, name, event, or decision (not vague hedges like "rates may rise").
Step 2: Count how many of those claims have an inline citation marker like [1], [2], [3] immediately after them.

Respond with ONLY this JSON, no other text:
{{"total_claims": <int>, "claims_with_citations": <int>, "citation_rate": <float 0-1>}}



## 6. Run the three experiments

Each experiment runs one pipeline over the full dataset with all three evaluators applied. `max_concurrency` sets the parallelism within an experiment, so its wall-clock time is roughly that of the slowest single item.

In [12]:
dataset = langfuse.get_dataset(DATASET_NAME)
n_items = len(list(dataset.items))
RUN_TAG = os.environ.get("RUN_TAG", "v1")  # bump this (or set $RUN_TAG) between rerun attempts
ANSWER_MODEL = os.environ.get("ANSWER_MODEL", "claude-opus-4-7")
print(f"Dataset '{DATASET_NAME}': {n_items} items.  RUN_TAG = '{RUN_TAG}'")

Dataset 'brave-finance-eval': 5 items.  RUN_TAG = 'v1'


In [13]:
# Experiment 1: baseline — LLM only, no retrieval
baseline_result = dataset.run_experiment(
    name=f"baseline-{RUN_TAG}",
    description="LLM only, no retrieval. Hallucination baseline.",
    task=baseline_task,
    evaluators=EVALUATORS,
    max_concurrency=5,
    metadata={"model": ANSWER_MODEL, "pipeline": "baseline"},
)
print(baseline_result.format())

Individual Results: Hidden (5 items)
💡 Set include_item_results=True to view them

──────────────────────────────────────────────────
🧪 Experiment: baseline-v1
📋 Run name: baseline-v1 - 2026-06-04T08:13:41.098331Z - LLM only, no retrieval. Hallucination baseline.
5 items
Evaluations:
  • citation_rate
  • latency_ms
  • factuality

Average Scores:
  • citation_rate: 0.000
  • latency_ms: 6367.320
  • factuality: 1.000

🔗 Dataset Run:
   http://localhost:3005/project/brave-demo/datasets/cmpp8jiub0006jp07ds10vq9z/runs/f2133c90-f8ba-4af1-9e85-64738da531d4


In [14]:
# Experiment 2: diy_rag — manual RAG pipeline
diy_result = dataset.run_experiment(
    name=f"diy-rag-{RUN_TAG}",
    description=(
        "Manual RAG: Brave Web Search + trafilatura extraction + paragraph chunking + "
        "sentence-transformers embeddings + FAISS retrieval. Same citation-forcing prompt."
    ),
    task=diy_task,
    evaluators=EVALUATORS,
    max_concurrency=3,  # local embedder + concurrent fetches: less is more
    metadata={
        "model": ANSWER_MODEL,
        "pipeline": "diy_rag",
        "embedding_model": "all-MiniLM-L6-v2",
        "vector_store": "faiss-IndexFlatIP",
    },
)
print(diy_result.format())

Individual Results: Hidden (5 items)
💡 Set include_item_results=True to view them

──────────────────────────────────────────────────
🧪 Experiment: diy-rag-v1
📋 Run name: diy-rag-v1 - 2026-06-04T08:14:28.832386Z - Manual RAG: Brave Web Search + trafilatura extraction + paragraph chunking + sentence-transformers embeddings + FAISS retrieval. Same citation-forcing prompt.
5 items
Evaluations:
  • citation_rate
  • latency_ms
  • factuality

Average Scores:
  • citation_rate: 0.854
  • latency_ms: 13197.420
  • factuality: 4.800

🔗 Dataset Run:
   http://localhost:3005/project/brave-demo/datasets/cmpp8jiub0006jp07ds10vq9z/runs/320853ec-bc92-4c2e-90f4-ee984622a164


In [15]:
# Experiment 3: brave-search-api — Brave LLM Context
brave_search_api_result = dataset.run_experiment(
    name=f"brave-search-api-{RUN_TAG}",
    description="Brave LLM Context endpoint + citation-forcing prompt.",
    task=brave_search_api_task,
    evaluators=EVALUATORS,
    max_concurrency=5,
    metadata={"model": ANSWER_MODEL, "pipeline": "brave-search-api", "freshness": "pm"},
)
langfuse.flush()
print(brave_search_api_result.format())

Individual Results: Hidden (5 items)
💡 Set include_item_results=True to view them

──────────────────────────────────────────────────
🧪 Experiment: brave-search-api-v1
📋 Run name: brave-search-api-v1 - 2026-06-04T08:15:58.265008Z - Brave LLM Context endpoint + citation-forcing prompt.
5 items
Evaluations:
  • citation_rate
  • latency_ms
  • factuality

Average Scores:
  • citation_rate: 0.910
  • latency_ms: 6931.520
  • factuality: 5.000

🔗 Dataset Run:
   http://localhost:3005/project/brave-demo/datasets/cmpp8jiub0006jp07ds10vq9z/runs/287f2835-a619-4b75-a194-167a18ced521


## 7. Comparison view in LangFuse

Open LangFuse → Datasets → `brave-finance-eval` → Runs → select all three → **Compare**.

What the scores show:

- **`factuality`** — baseline sits at the floor (ungrounded, ~1/5); both retrieval pipelines score near the top, with claims backed by their cited sources.
- **`citation_rate`** — baseline ~0 (no sources to cite); diy_rag and brave-search-api both high, as the citation-forcing prompt requires.
- **`latency_ms`** — brave-search-api is the fastest of the grounded pipelines; diy_rag is noticeably slower, carrying the overhead of its local embedder and per-page fetches.

View individual traces to inspect the full chain (retrieval → LLM call → cited answer).


In [16]:
host = os.environ.get("LANGFUSE_HOST", "http://localhost:3005")
print(f"Open this URL in your browser:\n  {host}\n")
print("Then navigate: Datasets → brave-finance-eval → Runs → select all three → Compare")

Open this URL in your browser:
  http://localhost:3005

Then navigate: Datasets → brave-finance-eval → Runs → select all three → Compare


## 8. Code contrast

The scores show `brave-search-api` matching or beating the alternatives on quality while running faster than `diy_rag`. This section looks at the engineering cost behind those numbers.

In [17]:
# Show the dependency declarations from pyproject.toml.
# TOML preserves the inline comments grouping shared vs DIY-only deps.
import pathlib
import re

text = pathlib.Path("pyproject.toml").read_text()
match = re.search(r"(dependencies = \[.*?\])", text, re.DOTALL)
print(match.group(1) if match else text)

dependencies = [
    # Shared by all three pipelines
    "anthropic>=0.40.0",
    "langfuse>=4.0.0",
    "requests>=2.32.0",
    # DIY RAG pipeline only — these are what Brave's LLM Context endpoint replaces
    "trafilatura>=1.12.0",
    "sentence-transformers>=2.7.0",
    "faiss-cpu>=1.8.0",
    "numpy>=1.26.0",
    # Notebook surface
    "jupyterlab>=4.0.0",
    "ipywidgets>=8.1.8",
]


In [18]:
# The DIY pipeline — eight numbered steps, three additional heavy dependencies.
# This is what Brave's LLM Context endpoint collapses into a single API call.
print(inspect.getsource(diy_rag_pipeline))

def diy_rag_pipeline(question: str) -> dict:
    """End-to-end manual RAG. The thing Brave LLM Context replaces with one call."""

    # 1. Search
    search_results = brave_web_search(question, count=10)
    if not search_results:
        return _empty("No search results returned.")

    # 2 + 3. Fetch + extract (concurrent)
    urls = [r["url"] for r in search_results[:8]]
    url_to_title = {r["url"]: r["title"] for r in search_results[:8]}
    extracted = fetch_all(urls)
    if not extracted:
        return _empty("All fetches failed or returned no extractable content.")

    # 4. Chunk every page
    chunks: list[dict] = []
    for url, content in extracted.items():
        for piece in chunk_text(content):
            chunks.append({"text": piece, "url": url, "title": url_to_title.get(url, "")})
    if not chunks:
        return _empty("No chunks produced from extracted content.")

    # 5 + 6. Embed + index
    chunk_vectors = embed([c["text"] for c in chunks])
    index = build

In [19]:
# The brave-search-api pipeline — ~20 lines of orchestration, one external API call to Brave.
# This is what the entire DIY stack above collapses into.
print(inspect.getsource(brave_search_api_pipeline))

def brave_search_api_pipeline(question: str) -> dict:
    """Brave LLM Context -> LLM with citation enforcement."""
    chunks = fetch_brave_context(question)
    if not chunks:
        return {
            "answer": "I cannot verify this from the provided sources.",
            "sources": [],
            "chunks_returned": 0,
        }

    sources_text = format_sources(chunks)
    answer = generate(CITATION_SYSTEM.format(sources=sources_text), question)
    return {
        "answer": answer,
        "sources": [{"n": c["n"], "url": c["url"], "title": c["title"]} for c in chunks],
        "sources_text": sources_text,
        "chunks_returned": len(chunks),
    }



## 9. Takeaways

* **Quality.** Grounding pays off: both retrieval pipelines cite specific facts and refuse the unanswerable question cleanly, while the ungrounded `baseline` hedges or invents specifics with nothing to back them.
* **Infrastructure.** Between the two grounded pipelines the cost is lopsided. `brave-search-api` is ~20 lines, one API call, and two dependencies; `diy_rag` is ~290 lines, eight steps, six dependencies, and ~2× slower — and the extra machinery buys no quality gain. Brave's LLM Context endpoint collapses an entire retrieval stack into a single HTTP call.
* **Caveat.** A demonstration, not a benchmark — five questions, non-deterministic LLM-as-judge scoring. For rigor, run several variants (different `RUN_TAG`s) and average.
* **Keep exploring.** Re-run with a new `RUN_TAG` after tuning brave (`count`, `freshness`, `context_threshold_mode`) or diy_rag (chunk size, embedder, `k`); add `QUESTIONS`; or swap `ANSWER_MODEL` to compare Claude models with retrieval held fixed.